In [1]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ============================================================================
# 1. OPTIMIZED DATASET WITH NORMALIZATION
# ============================================================================
class EmbeddingDataset(Dataset):
    def __init__(self, data, scaler=None, is_test=False, fit_scaler=False):
        self.data = data
        self.is_test = is_test
        
        # Extract features
        features = []
        for record in data:
            feat = record['image_embedding'] + record['text_embedding']
            features.append(feat)
        
        features = np.array(features)
        
        # Normalize features
        if fit_scaler:
            self.scaler = StandardScaler()
            features = self.scaler.fit_transform(features)
        elif scaler is not None:
            self.scaler = scaler
            features = self.scaler.transform(features)
        else:
            self.scaler = None
        
        self.features = features
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features[idx])
        
        if self.is_test:
            return features, self.data[idx]['id']
        else:
            label = torch.LongTensor([self.data[idx]['label']])[0]
            return features, label

# ============================================================================
# 2. IMPROVED MODEL ARCHITECTURE WITH BETTER REGULARIZATION
# ============================================================================
class OptimizedEarlyFusionMLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=[512, 256, 128, 64], 
                 dropout_rate=0.4):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate if i < len(hidden_dims) - 1 else dropout_rate * 0.5)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 2))
        self.model = nn.Sequential(*layers)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ============================================================================
# 3. TRAINING FUNCTION WITH CLASS BALANCING
# ============================================================================
def train_model(train_loader, val_loader, device, config, class_weights=None):
    """
    Train model with class weight balancing for better minority class performance
    """
    model = OptimizedEarlyFusionMLP(
        hidden_dims=config['hidden_dims'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # Loss function with class weights for imbalanced data
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))
    else:
        criterion = nn.CrossEntropyLoss()
    
    # Optimizer with weight decay (L2 regularization)
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',  # maximize F1
        factor=0.5, 
        patience=5,
        min_lr=1e-6
    )
    
    best_f1 = 0
    patience_counter = 0
    train_losses = []
    val_f1_scores = []
    
    for epoch in range(config['num_epochs']):
        # Training phase
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        
        # Validation phase
        model.eval()
        val_preds = []
        val_labels = []
        val_loss = 0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        train_losses.append(avg_train_loss)
        val_f1_scores.append(val_f1)
        
        # Get current learning rate before scheduler step
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update learning rate
        old_lr = current_lr
        scheduler.step(val_f1)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['num_epochs']}")
            print(f"  Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
            print(f"  LR: {new_lr:.6f}")
            if new_lr < old_lr:
                print(f"  → Learning rate reduced from {old_lr:.6f} to {new_lr:.6f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
                'config': config
            }, 'best_model.pth')
            patience_counter = 0
            print(f"  ✓ New best F1: {best_f1:.4f}")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    print(f"\nTraining completed. Best Val F1: {best_f1:.4f}")
    
    # Print final classification report
    print("\nFinal Validation Classification Report:")
    print(classification_report(val_labels, val_preds, 
                                target_names=['Not Important', 'Important']))
    
    return model, best_f1, train_losses, val_f1_scores

# ============================================================================
# 4. IMPROVED HYPERPARAMETER CONFIGURATIONS
# ============================================================================
def hyperparameter_search(train_data, device):
    """
    Optimized configurations to improve test F1 from 0.4873
    Focus on better generalization and minority class performance
    """
    
    # Calculate class weights for imbalanced data
    labels = [d['label'] for d in train_data]
    class_counts = [labels.count(0), labels.count(1)]
    total = len(labels)
    class_weights = [total / (2 * c) for c in class_counts]
    print(f"Class weights: {class_weights}")
    
    param_grid = [
        # ===== BASELINE: Original best config =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.535,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20,
            'use_class_weights': True
        },
        
        # ===== HIGHER DROPOUT FOR BETTER GENERALIZATION =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.58,  # Higher dropout
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20,
            'use_class_weights': True
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.6,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20,
            'use_class_weights': True
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 2e-5,  # Higher weight decay
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20,
            'use_class_weights': True
        },
        
        # ===== SIMPLER ARCHITECTURES (LESS OVERFITTING) =====
        {
            'hidden_dims': [512, 256],  # Simpler 2-layer
            'dropout_rate': 0.5,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20,
            'use_class_weights': True
        },
        {
            'hidden_dims': [640, 320, 160],  # Smaller network
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20,
            'use_class_weights': True
        },
        
        # ===== STRONGER REGULARIZATION =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.6,
            'learning_rate': 0.0008,  # Lower LR
            'weight_decay': 3e-5,  # Higher weight decay
            'batch_size': 32,
            'num_epochs': 130,
            'early_stopping_patience': 18,  # Earlier stopping
            'use_class_weights': True
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.58,
            'learning_rate': 0.0009,
            'weight_decay': 2e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 18,
            'use_class_weights': True
        },
        
        # ===== SMALLER BATCH SIZES FOR BETTER GENERALIZATION =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.58,
            'learning_rate': 0.0008,
            'weight_decay': 1.5e-5,
            'batch_size': 16,  # Smaller batch
            'num_epochs': 120,
            'early_stopping_patience': 18,
            'use_class_weights': True
        },
        {
            'hidden_dims': [640, 320, 160],
            'dropout_rate': 0.55,
            'learning_rate': 0.0009,
            'weight_decay': 2e-5,
            'batch_size': 24,
            'num_epochs': 120,
            'early_stopping_patience': 18,
            'use_class_weights': True
        },
        
        # ===== CROSS-VALIDATION WINNERS (ENSEMBLE-READY) =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.57,
            'learning_rate': 0.0009,
            'weight_decay': 1.5e-5,
            'batch_size': 28,
            'num_epochs': 120,
            'early_stopping_patience': 18,
            'use_class_weights': True
        },
        {
            'hidden_dims': [700, 350, 175],
            'dropout_rate': 0.56,
            'learning_rate': 0.0009,
            'weight_decay': 1.8e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 18,
            'use_class_weights': True
        },
        {
            'hidden_dims': [800, 400, 200],
            'dropout_rate': 0.58,
            'learning_rate': 0.0008,
            'weight_decay': 2e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 18,
            'use_class_weights': True
        },
        
        # ===== CONSERVATIVE APPROACHES =====
        {
            'hidden_dims': [512, 256, 128],
            'dropout_rate': 0.52,
            'learning_rate': 0.0008,
            'weight_decay': 2e-5,
            'batch_size': 32,
            'num_epochs': 110,
            'early_stopping_patience': 16,
            'use_class_weights': True
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.62,  # Very high dropout
            'learning_rate': 0.0009,
            'weight_decay': 2.5e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 17,
            'use_class_weights': True
        },
    ]
    
    best_config = None
    best_score = 0
    results_log = []
    
    for i, config in enumerate(param_grid):
        print(f"\n{'='*60}")
        print(f"Testing configuration {i+1}/{len(param_grid)}")
        print(f"{'='*60}")
        
        use_weights = config.pop('use_class_weights', False)
        weights = class_weights if use_weights else None
        
        print(config)
        
        # Split data
        train_split, val_split = train_test_split(
            train_data, test_size=0.2, random_state=42, 
            stratify=[d['label'] for d in train_data]
        )
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_split, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_split, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config, weights)
        
        # Log results
        results_log.append({
            'config_idx': i + 1,
            'f1_score': f1,
            'config': config
        })
        
        if f1 > best_score:
            best_score = f1
            best_config = config
            print(f"\n✓ New best configuration! F1: {f1:.4f}")
    
    # Print summary of all results
    print(f"\n{'='*60}")
    print("SUMMARY OF ALL CONFIGURATIONS:")
    print(f"{'='*60}")
    results_log.sort(key=lambda x: x['f1_score'], reverse=True)
    for i, result in enumerate(results_log[:5]):  # Top 5
        print(f"\n{i+1}. F1: {result['f1_score']:.4f}")
        print(f"   Config {result['config_idx']}: {result['config']}")
    
    print(f"\n{'='*60}")
    print("BEST CONFIGURATION:")
    print(best_config)
    print(f"Best F1 Score: {best_score:.4f}")
    print(f"{'='*60}")
    
    return best_config, class_weights

# ============================================================================
# 5. ENSEMBLE PREDICTION FOR BETTER ROBUSTNESS
# ============================================================================
def train_ensemble(train_data, device, configs, class_weights):
    """
    Train multiple models and ensemble predictions
    """
    models = []
    scalers = []
    
    for i, config in enumerate(configs):
        print(f"\n{'='*60}")
        print(f"Training ensemble model {i+1}/{len(configs)}")
        print(f"{'='*60}")
        
        # Use different random split for diversity
        train_split, val_split = train_test_split(
            train_data, test_size=0.2, random_state=42 + i, 
            stratify=[d['label'] for d in train_data]
        )
        
        train_dataset = EmbeddingDataset(train_split, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_split, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
        
        model, f1, _, _ = train_model(train_loader, val_loader, device, config, class_weights)
        
        # Load best model
        checkpoint = torch.load('best_model.pth', weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        models.append(model)
        scalers.append(train_dataset.scaler)
        
        print(f"Model {i+1} F1: {f1:.4f}")
    
    return models, scalers

# ============================================================================
# 6. MAIN TRAINING PIPELINE
# ============================================================================
def main():
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load data
    print("\nLoading training data...")
    with open('train_part1.json', 'r') as f:
        data = json.load(f)
    
    print(f"Total samples: {len(data)}")
    labels = [d['label'] for d in data]
    print(f"Class distribution: 0={labels.count(0)}, 1={labels.count(1)}")
    print(f"Class imbalance ratio: {labels.count(0)/labels.count(1):.2f}:1")
    
    # Hyperparameter search
    best_config, class_weights = hyperparameter_search(data, device)
    
    # Train ensemble of top 3 configs for better performance
    print("\n" + "="*60)
    print("TRAINING ENSEMBLE OF TOP MODELS")
    print("="*60)
    
    # Create ensemble configs (slight variations)
    ensemble_configs = [
        best_config,  # Best config
        {**best_config, 'dropout_rate': best_config['dropout_rate'] + 0.02},
        {**best_config, 'dropout_rate': best_config['dropout_rate'] - 0.02},
    ]
    
    models, scalers = train_ensemble(data, device, ensemble_configs, class_weights)
    
    # Make predictions on test set with ensemble
    print("\nMaking ensemble predictions on test set...")
    with open('test.json', 'r') as f:
        test_data = json.load(f)
    
    all_predictions = []
    test_ids = None
    
    for model, scaler in zip(models, scalers):
        model.eval()
        test_dataset = EmbeddingDataset(test_data, scaler=scaler, is_test=True)
        test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
        
        predictions = []
        ids = []
        
        with torch.no_grad():
            for features, batch_ids in test_loader:
                features = features.to(device)
                outputs = model(features)
                probs = torch.softmax(outputs, dim=1).cpu().numpy()
                predictions.append(probs)
                
                if isinstance(batch_ids, torch.Tensor):
                    ids.extend(batch_ids.cpu().numpy().tolist())
                else:
                    ids.extend(batch_ids)
        
        all_predictions.append(np.vstack(predictions))
        if test_ids is None:
            test_ids = ids
    
    # Average ensemble predictions
    avg_probs = np.mean(all_predictions, axis=0)
    final_predictions = np.argmax(avg_probs, axis=1).tolist()
    
    # Create submission
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': final_predictions
    })
    submission.to_csv('ann_v10.csv', index=False)
    print("\n✓ Submission file created: submission_improved.csv")
    print(f"Predictions: 0={final_predictions.count(0)}, 1={final_predictions.count(1)}")
    
    print(f"\n{'='*60}")
    print("IMPROVEMENTS APPLIED:")
    print(f"{'='*60}")
    print("✓ Class weight balancing for minority class")
    print("✓ Increased dropout for better generalization")
    print("✓ Stronger weight decay regularization")
    print("✓ Ensemble of 3 models for robustness")
    print("✓ Earlier stopping to prevent overfitting")
    print(f"Expected test F1 improvement: > 0.55 (from 0.4873)")

if __name__ == "__main__":
    main()

Using device: cuda

Loading training data...
Total samples: 1530
Class distribution: 0=1326, 1=204
Class imbalance ratio: 6.50:1
Class weights: [0.5769230769230769, 3.75]

Testing configuration 1/15
{'hidden_dims': [768, 384, 192], 'dropout_rate': 0.535, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 32, 'num_epochs': 120, 'early_stopping_patience': 20}
Epoch 1/120
  Train Loss: 4.3948, Train F1: 0.5076
  Val Loss: 1.8753, Val F1: 0.5440
  LR: 0.001000
  ✓ New best F1: 0.5440
  ✓ New best F1: 0.6009
Epoch 5/120
  Train Loss: 2.5815, Train F1: 0.6388
  Val Loss: 1.6918, Val F1: 0.6168
  LR: 0.001000
  ✓ New best F1: 0.6168
  ✓ New best F1: 0.6679
  ✓ New best F1: 0.6745
  ✓ New best F1: 0.6915
Epoch 10/120
  Train Loss: 1.2748, Train F1: 0.7311
  Val Loss: 2.6654, Val F1: 0.6632
  LR: 0.001000
  ✓ New best F1: 0.6926
Epoch 15/120
  Train Loss: 0.9070, Train F1: 0.7813
  Val Loss: 2.5574, Val F1: 0.6331
  LR: 0.001000
Epoch 20/120
  Train Loss: 0.5005, Train F1: 0.8517
  Va

In [ ]:
# Run this in a new cell first:
import sys
import importlib

# Remove torch from cache
modules_to_remove = [k for k in sys.modules.keys() if 'torch' in k]
for module in modules_to_remove:
    del sys.modules[module]

# Now import fresh
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")